# Notebook 03 — ML Basics with PyTorch

This notebook trains a small neural network to act as a **surrogate model** for the
IceCube neutrino flux — the same pattern used in
[Prometheus's `olympus` module](https://github.com/Harvard-Neutrino/prometheus/tree/main/olympus)
to replace expensive optical simulations with fast network calls.

You will:
1. Create tensors and move them to a device
2. Define a `torch.nn.Module`
3. Write a training loop
4. Evaluate and compare to the true analytical flux

> **Prerequisite**: `pip install torch`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam


def _find_style() -> Path | None:
    path = Path.cwd()
    for _ in range(6):
        candidate = path / "examples" / "plotting" / "styles" / "nestling.mplstyle"
        if candidate.exists():
            return candidate
        path = path.parent
    return None


_style = _find_style()
if _style:
    plt.style.use(str(_style))
    print(f"Style loaded: {_style}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} — device: {device}")

## 1. Tensors

A PyTorch `Tensor` is a GPU-ready NumPy array that can track gradients.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(5)
c = torch.randn(3, 4)   # shape (3, 4), random normal

print("shape :", c.shape)
print("dtype :", c.dtype)
print("device:", c.device)

# Convert to/from NumPy (CPU only)
arr = c.numpy()
c2  = torch.from_numpy(arr)
assert torch.allclose(c, c2)
print("NumPy round-trip passed.")

## 2. Training data

We generate samples from the IceCube power-law flux and work in **log-log space**
(the relationship is exactly linear there), making it easy for the network to learn.

In [ ]:
GAMMA = 2.37
NORM  = 1.66e-18
E_REF = 1e5

def true_flux(E):
    return NORM * (E / E_REF) ** (-GAMMA)

# 2 000 log-spaced energies from 1 TeV to 10 PeV
E_train = np.logspace(3, 7, 2000)
phi_train = true_flux(E_train)

log_E   = torch.tensor(np.log10(E_train / E_REF), dtype=torch.float32).unsqueeze(1)
log_phi = torch.tensor(np.log10(phi_train),        dtype=torch.float32).unsqueeze(1)

log_E   = log_E.to(device)
log_phi = log_phi.to(device)

print(f"Input  shape: {log_E.shape}   range [{log_E.min():.1f}, {log_E.max():.1f}]")
print(f"Target shape: {log_phi.shape}  range [{log_phi.min():.1f}, {log_phi.max():.1f}]")

## 3. Defining the model

A `torch.nn.Module` subclass with two hidden layers.
The class encapsulates all parameters; `forward` defines the computation.

In [ ]:
class FluxSurrogate(nn.Module):
    """
    Predicts log10(flux) from log10(E / E_ref).

    A minimal surrogate model — the same design pattern as Prometheus's
    olympus module, which surrogate-models full photon propagation.
    """

    def __init__(self, n_hidden: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


model = FluxSurrogate(n_hidden=64).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {n_params}")

## 4. Training loop

Four steps every iteration:

1. `optimizer.zero_grad()` — clear old gradients
2. `model(log_E)` — forward pass
3. `loss.backward()` — backpropagation (compute gradients)
4. `optimizer.step()` — update weights

In [ ]:
optimizer    = Adam(model.parameters(), lr=1e-3)
loss_fn      = nn.MSELoss()
loss_history = []

for epoch in range(2000):
    optimizer.zero_grad()              # 1. clear gradients
    pred = model(log_E)                # 2. forward pass
    loss = loss_fn(pred, log_phi)     # 3. loss
    loss.backward()                    # 4. backprop
    optimizer.step()                   # 5. update weights
    loss_history.append(loss.item())

    if epoch % 400 == 0:
        print(f"Epoch {epoch:5d}  MSE loss = {loss.item():.4e}")

print("\nTraining complete.")

## 5. Evaluation

In [ ]:
model.eval()

E_test = np.logspace(3, 7, 300)
log_E_test = torch.tensor(np.log10(E_test / E_REF), dtype=torch.float32).unsqueeze(1).to(device)

with torch.no_grad():   # disable gradient tracking for inference
    log_phi_pred = model(log_E_test).cpu().numpy().squeeze()

phi_pred = 10 ** log_phi_pred
phi_true = true_flux(E_test)

residuals = np.abs(phi_pred - phi_true) / phi_true * 100
print(f"Max residual : {residuals.max():.3f} %")
print(f"Mean residual: {residuals.mean():.4f} %")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))

# Spectrum comparison
ax = axes[0]
ax.loglog(E_test, E_test**2 * phi_true,  label="Analytical", lw=2)
ax.loglog(E_test, E_test**2 * phi_pred,  label="Surrogate",  lw=1.5, ls="--")
ax.set_xlabel(r"$E$ [GeV]")
ax.set_ylabel(r"$E^2\,\Phi$  [GeV cm$^{-2}$ s$^{-1}$ sr$^{-1}$]")
ax.set_title("Flux surrogate")
ax.legend()

# Training loss
ax = axes[1]
ax.semilogy(loss_history)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Training convergence")

fig.tight_layout()
fig.savefig("/tmp/flux_surrogate.pdf")
plt.show()

## 6. Saving and loading

In [ ]:
torch.save(model.state_dict(), "/tmp/flux_surrogate.pt")

loaded = FluxSurrogate(n_hidden=64).to(device)
loaded.load_state_dict(torch.load("/tmp/flux_surrogate.pt", weights_only=True))
loaded.eval()

with torch.no_grad():
    pred_loaded = loaded(log_E_test).cpu().numpy().squeeze()

assert np.allclose(log_phi_pred, pred_loaded)
print("Save/load round-trip passed.")

## Summary

| Concept | Where it appears |
|---------|------------------|
| Tensors | `log_E`, `log_phi`, `.to(device)` |
| `nn.Module` | `FluxSurrogate` with `__init__` and `forward` |
| Training loop | `zero_grad → forward → backward → step` |
| Evaluation | `model.eval()` + `torch.no_grad()` |
| Save/load | `torch.save` / `load_state_dict` |

See the [ML Basics lesson](../../../docs/tracks/05-coding/lesson-07.md) and
[Prometheus olympus](https://github.com/Harvard-Neutrino/prometheus/tree/main/olympus)
for a real surrogate model built on these same principles.